<a href="https://colab.research.google.com/github/thienandinh29/nghiemcuukhoahocsgu2026/blob/main/raw_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Exploratory Data Analysis

**Goal.** Validate the raw dataset before modelling: shape, missing values, class balance and price sanity.

**Input:** `sp500_supervised_data.csv`

## 1. Setup

In [ ]:
# ── Shared figure style ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

COLORS = {
    "navy":  "#1F3A5F",   # primary / price
    "teal":  "#2A9D8F",   # positive / strategy
    "amber": "#E9A23B",   # secondary series
    "red":   "#C8553D",   # negative / drawdown
    "grey":  "#6B7280",   # benchmark / text
}
SERIES = [COLORS["teal"], COLORS["amber"], COLORS["red"], COLORS["navy"]]

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 10, "axes.labelcolor": COLORS["grey"],
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#D1D5DB", "axes.prop_cycle": plt.cycler(color=SERIES),
    "axes.grid": True, "grid.color": "#E5E7EB", "grid.linewidth": 0.8,
    "xtick.color": COLORS["grey"], "ytick.color": COLORS["grey"],
    "legend.frameon": False, "legend.fontsize": 9,
})

## 2. Load and profile

In [ ]:
import pandas as pd

# parse_dates=True makes the "Date" index a real datetime
df = pd.read_csv("sp500_supervised_data.csv", index_col=0, parse_dates=True)

print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print("\nMissing values per column:")
print(df.isnull().sum())

up_days   = int((df["Target"] == 1).sum())
down_days = int((df["Target"] == 0).sum())
print(f"\nUp days (1):   {up_days:,}")
print(f"Down days (0): {down_days:,}")
print(f"Up ratio:      {up_days / (up_days + down_days):.2%}   (healthy range ≈ 45–55%)")

## 3. Visual checks

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]})

# A · Closing price — look for gaps or zero values
ax1.plot(df.index, df["Close"], color=COLORS["navy"], linewidth=1.4)
ax1.fill_between(df.index, df["Close"], df["Close"].min(), color=COLORS["navy"], alpha=0.08)
ax1.set(title="S&P 500 closing price, 2010–2025", xlabel="", ylabel="Index level")

# B · Class balance
counts = df["Target"].value_counts().sort_index()
bars = ax2.bar(["Down (0)", "Up (1)"], counts.values, color=[COLORS["red"], COLORS["teal"]], width=0.55)
for b, v in zip(bars, counts.values):
    ax2.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / counts.sum():.1%})",
             ha="center", va="bottom", fontsize=10, color=COLORS["grey"])
ax2.set(title="Class balance", ylabel="Trading days", ylim=(0, counts.max() * 1.18))
ax2.grid(axis="x", visible=False)

fig.tight_layout()
plt.show()

## 4. Descriptive statistics

Check `min` / `max` for impossible values (e.g. zero volume days).

In [ ]:
df[["Close", "Volume"]].describe().round(2)